In [2]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from collections import defaultdict
from typing import Union, Sequence, Tuple, List, Optional
from itertools import chain

In [2]:


def all_leaf_dirs_have_mk(
    root_dir: Union[str, Path],
    filename: str = "mk.csv",
    *,
    return_missing: bool = False
) -> Union[bool, Tuple[bool, Sequence[Path]]]:
    """
    Recursively ensure that every *leaf* directory (one with no sub‑directories)
    beneath *root_dir* contains ``filename``.

    Parameters
    ----------
    root_dir : str | pathlib.Path
        Directory at which to start the walk.
    filename : str, default "mk.csv"
        Required file name (case‑sensitive).
    return_missing : bool, default False
        If True, return a tuple ``(ok, missing)`` where *missing* lists the
        offending leaf directories.

    Returns
    -------
    bool
        True  – every leaf directory contained *filename*
        False – at least one leaf directory lacked *filename*
    tuple[bool, list[pathlib.Path]]  (if *return_missing* is True)
    """
    root = Path(root_dir).expanduser().resolve()
    if not root.is_dir():
        raise NotADirectoryError(f"{root} is not a directory")

    missing: List[Path] = []

    # Examine *only* directories, starting with root, then all contents.
    for current_dir in chain([root], root.rglob("*")):
        if not current_dir.is_dir():
            continue

        # List immediate sub‑directories of the current directory
        subdirs = [d for d in current_dir.iterdir() if d.is_dir()]

        # A leaf directory has no sub‑directories
        if not subdirs and not (current_dir / filename).is_file():
            missing.append(current_dir)

    ok = len(missing) == 0
    return (ok, missing) if return_missing else ok

test = all_leaf_dirs_have_mk("./checkpoints_exp", return_missing=True)
print(test)

(True, [])


In [1]:
def get_norm_scores():
    dis_pop_df = pd.read_csv("./datasets_results/dis_pop.csv")
    dispatching_df = pd.read_csv("./datasets_results/dispatching.csv")
    population_df = pd.read_csv("./datasets_results/population.csv")
    random_df = pd.read_csv("./datasets_results/random.csv")
    datasets = dis_pop_df["dataset"].tolist()
    dict_norm = {
        "dis_pop": {},
        "dispatching": {},
        "population": {},
        "random": {}
    }
    for d in datasets:
        min_gap_dis_pop = dis_pop_df[dis_pop_df["dataset"] == d]["min_gap"].iloc[0]

        max_gap_dis_pop = dis_pop_df[dis_pop_df["dataset"] == d]["max_gap"].iloc[0]
        min_make_dis_pop = dis_pop_df[dis_pop_df["dataset"] == d]["min_makespan"].iloc[0]
        max_make_dis_pop = dis_pop_df[dis_pop_df["dataset"] == d]["max_makespan"].iloc[0]
        min_gap_dispatching = dispatching_df[dispatching_df["dataset"] == d]["min_gap"].iloc[0]
        max_gap_dispatching = dispatching_df[dispatching_df["dataset"] == d]["max_gap"].iloc[0]
        min_make_dispatching = dispatching_df[dispatching_df["dataset"] == d]["min_makespan"].iloc[0]
        max_make_dispatching = dispatching_df[dispatching_df["dataset"] == d]["max_makespan"].iloc[0]

        min_gap_population = population_df[population_df["dataset"] == d]["min_gap"].iloc[0]
        max_gap_population = population_df[population_df["dataset"] == d]["max_gap"].iloc[0]
        min_make_population = population_df[population_df["dataset"] == d]["min_makespan"].iloc[0]
        max_make_population = population_df[population_df["dataset"] == d]["max_makespan"].iloc[0]

        min_gap_random = random_df[random_df["dataset"] == d]["min_gap"].iloc[0]
        max_gap_random = random_df[random_df["dataset"] == d]["max_gap"].iloc[0]
        min_make_random = random_df[random_df["dataset"] == d]["min_makespan"].iloc[0]
        max_make_random = random_df[random_df["dataset"] == d]["max_makespan"].iloc[0]
        if d == "Brandimarte":
            name = "mk"
        elif d == "Hurink_edata":
            name = "edata"
        elif d == "Hurink_rdata":
            name = "rdata"
        elif d == "Hurink_vdata":
            name = "vdata"
        else:
            name = d
        dict_norm["dis_pop"][name] = {
            "min_gap": float(min_gap_dis_pop),
            "max_gap": float(max_gap_dis_pop),
            "min_makespan": float(min_make_dis_pop),
            "max_makespan": float(max_make_dis_pop)
        }
        dict_norm["dispatching"][name] = {
            "min_gap": float(min_gap_dispatching),
            "max_gap": float(max_gap_dispatching),
            "min_makespan": float(min_make_dispatching),
            "max_makespan": float(max_make_dispatching)
        }
        dict_norm["population"][name] = {
            "min_gap": float(min_gap_population),
            "max_gap": float(max_gap_population),
            "min_makespan": float(min_make_population),
            "max_makespan": float(max_make_population)

        }
        dict_norm["random"][name] = {
            "min_gap": float(min_gap_random),
            "max_gap": float(max_gap_random),
            "min_makespan": float(min_make_random),
            "max_makespan": float(max_make_random)
        }



        # print(d, min_gap, max_gap)
    return dict_norm

def get_ref_score(score, min_score, max_score):
    new_min_score = -max_score
    new_max_score = -min_score
    score = -score
    return (score - new_min_score) / (new_max_score - new_min_score)

def get_results_comb_bench(folder_path: str, dataset: str, norm_scores: dict, n_j, n_m):

    edata_df = pd.read_csv(os.path.join(folder_path, "edata.csv"))
    vdata_df = pd.read_csv(os.path.join(folder_path, "vdata.csv"))
    rdata_df = pd.read_csv(os.path.join(folder_path, "rdata.csv"))
    mk_df = pd.read_csv(os.path.join(folder_path, "mk.csv"))
    sd1_df = pd.read_csv(os.path.join(folder_path, "sd1.csv"))
    norm_scores_d = norm_scores[dataset]
    min_gap_edata = norm_scores_d["edata"]["min_gap"]
    max_gap_edata = norm_scores_d["edata"]["max_gap"]
    min_makespan_edata = norm_scores_d["edata"]["min_makespan"]
    max_makespan_edata = norm_scores_d["edata"]["max_makespan"]
    min_gap_vdata = norm_scores_d["vdata"]["min_gap"]
    max_gap_vdata = norm_scores_d["vdata"]["max_gap"]
    min_makespan_vdata = norm_scores_d["vdata"]["min_makespan"]
    max_makespan_vdata = norm_scores_d["vdata"]["max_makespan"]

    min_gap_rdata = norm_scores_d["rdata"]["min_gap"]
    max_gap_rdata = norm_scores_d["rdata"]["max_gap"]
    min_makespan_rdata = norm_scores_d["rdata"]["min_makespan"]
    max_makespan_rdata = norm_scores_d["rdata"]["max_makespan"]
    min_gap_mk = norm_scores_d["mk"]["min_gap"]
    max_gap_mk = norm_scores_d["mk"]["max_gap"]
    min_makespan_mk = norm_scores_d["mk"]["min_makespan"]
    max_makespan_mk = norm_scores_d["mk"]["max_makespan"]
    min_gap_sd1 = norm_scores_d[f"SD1_{n_j}_{n_m}"]["min_gap"]
    max_gap_sd1 = norm_scores_d[f"SD1_{n_j}_{n_m}"]["max_gap"]
    min_makespan_sd1 = norm_scores_d[f"SD1_{n_j}_{n_m}"]["min_makespan"]
    max_makespan_sd1 = norm_scores_d[f"SD1_{n_j}_{n_m}"]["max_makespan"]



    edata_makespan_greedy = edata_df["makespan_det_offline"].mean()
    edata_makespan_greedy_norm = get_ref_score(edata_makespan_greedy, min_makespan_edata, max_makespan_edata)

    edata_gap_greedy = edata_df["gap_det_offline"].mean()
    edata_gap_greedy_norm = get_ref_score(edata_gap_greedy, min_gap_edata, max_gap_edata)

    edata_runtime_greedy = edata_df["runtime_det_offline"].mean()
    edata_makespan_samp = edata_df["makespan_stoch_offline"].mean()
    edata_makespan_samp_norm = get_ref_score(edata_makespan_samp, min_makespan_edata, max_makespan_edata)
    edata_gap_samp = edata_df["gap_stoch_offline"].mean()
    edata_gap_samp_norm = get_ref_score(edata_gap_samp, min_gap_edata, max_gap_edata)
    edata_runtime_samp = edata_df["runtime_stoch_offline"].mean()

    vdata_makespan_greedy = vdata_df["makespan_det_offline"].mean()
    vdata_norm_makespan_greedy = get_ref_score(vdata_makespan_greedy, min_makespan_vdata, max_makespan_vdata)
    vdata_gap_greedy = vdata_df["gap_det_offline"].mean()
    vdata_norm_gap_greedy = get_ref_score(vdata_gap_greedy, min_gap_vdata, max_gap_vdata)
    vdata_runtime_greedy = vdata_df["runtime_det_offline"].mean()

    vdata_makespan_samp = vdata_df["makespan_stoch_offline"].mean()
    vdata_makespan_samp_norm = get_ref_score(vdata_makespan_samp, min_makespan_vdata, max_makespan_vdata)
    vdata_runtime_samp = vdata_df["runtime_stoch_offline"].mean()

    vdata_gap_samp = vdata_df["gap_stoch_offline"].mean()
    vdata_gap_samp_norm = get_ref_score(vdata_gap_samp, min_gap_vdata, max_gap_vdata)

    rdata_makespan_greedy = rdata_df["makespan_det_offline"].mean()
    rdata_makespan_greedy_norm = get_ref_score(rdata_makespan_greedy, min_makespan_rdata, max_makespan_rdata)
    rdata_gap_greedy = rdata_df["gap_det_offline"].mean()
    rdata_gap_greedy_norm = get_ref_score(rdata_gap_greedy, min_gap_rdata, max_gap_rdata)
    rdata_runtime_greedy = rdata_df["runtime_det_offline"].mean()
    rdata_makespan_samp = rdata_df["makespan_stoch_offline"].mean()
    rdata_makespan_samp_norm = get_ref_score(rdata_makespan_samp, min_makespan_rdata, max_makespan_rdata)
    rdata_runtime_samp = rdata_df["runtime_stoch_offline"].mean()
    rdata_gap_samp = rdata_df["gap_stoch_offline"].mean()
    rdata_gap_samp_norm = get_ref_score(rdata_gap_samp, min_gap_rdata, max_gap_rdata)

    mk_makespan_greedy = mk_df["makespan_det_offline"].mean()
    mk_makespan_greedy_norm = get_ref_score(mk_makespan_greedy, min_makespan_mk, max_makespan_mk)
    mk_gap_greedy = mk_df["gap_det_offline"].mean()
    mk_gap_greedy_norm = get_ref_score(mk_gap_greedy, min_gap_mk, max_gap_mk)

    mk_runtime_greedy = mk_df["runtime_det_offline"].mean()
    mk_makespan_samp = mk_df["makespan_stoch_offline"].mean()
    mk_makespan_samp_norm = get_ref_score(mk_makespan_samp, min_makespan_mk, max_makespan_mk)
    mk_runtime_samp = mk_df["runtime_stoch_offline"].mean()
    mk_gap_samp = mk_df["gap_stoch_offline"].mean()
    mk_gap_samp_norm = get_ref_score(mk_gap_samp, min_gap_mk, max_gap_mk)

    sd1_makespan_greedy = sd1_df["makespan_det_offline"].mean()
    sd1_makespan_greedy_norm = get_ref_score(sd1_makespan_greedy, min_makespan_sd1, max_makespan_sd1)
    sd1_gap_greedy = sd1_df["gap_det_offline"].mean()
    sd1_gap_greedy_norm = get_ref_score(sd1_gap_greedy, min_gap_sd1, max_gap_sd1)
    sd1_runtime_greedy = sd1_df["runtime_det_offline"].mean()
    sd1_makespan_samp = sd1_df["makespan_stoch_offline"].mean()
    sd1_makespan_samp_norm = get_ref_score(sd1_makespan_samp, min_makespan_sd1, max_makespan_sd1)
    sd1_gap_samp = sd1_df["gap_stoch_offline"].mean()
    sd1_gap_samp_norm = get_ref_score(sd1_gap_samp, min_gap_sd1, max_gap_sd1)
    sd1_runtime_samp = sd1_df["runtime_stoch_offline"].mean()

    return edata_makespan_greedy, edata_makespan_greedy_norm, edata_gap_greedy, edata_gap_greedy_norm, edata_runtime_greedy, edata_makespan_samp, edata_makespan_samp_norm, edata_gap_samp, edata_gap_samp_norm, edata_runtime_samp, \
           vdata_makespan_greedy, vdata_norm_makespan_greedy, vdata_gap_greedy, vdata_norm_gap_greedy, vdata_runtime_greedy, vdata_makespan_samp, vdata_makespan_samp_norm, vdata_gap_samp, vdata_gap_samp_norm, vdata_runtime_samp, \
           rdata_makespan_greedy, rdata_makespan_greedy_norm, rdata_gap_greedy, rdata_gap_greedy_norm, rdata_runtime_greedy, rdata_makespan_samp, rdata_makespan_samp_norm, rdata_gap_samp, rdata_gap_samp_norm, rdata_runtime_samp,\
           mk_makespan_greedy, mk_makespan_greedy_norm, mk_gap_greedy, mk_gap_greedy_norm, mk_runtime_greedy, mk_makespan_samp, mk_makespan_samp_norm, mk_gap_samp, mk_gap_samp_norm,\
           mk_runtime_samp,\
           sd1_makespan_greedy, sd1_makespan_greedy_norm,\
           sd1_gap_greedy,\
           sd1_gap_greedy_norm,\
           sd1_runtime_greedy,\
           sd1_makespan_samp,\
           sd1_makespan_samp_norm,\
           sd1_gap_samp,\
           sd1_gap_samp_norm,\
           sd1_runtime_samp

def group_folders_by_prefix(parent: str | Path) -> dict[str, list[Path]]:
    """
    Traverse *parent* and group every immediate sub‑directory by the prefix
    that appears before the first hyphen in its name.

    Parameters
    ----------
    parent : str | Path
        Path to the directory whose immediate sub‑folders will be examined.

    Returns
    -------
    dict[str, list[pathlib.Path]]
        Mapping from discovered prefix to the corresponding folders.
    """
    parent = Path(parent).expanduser().resolve()
    if not parent.is_dir():
        raise NotADirectoryError(f"{parent} is not a directory.")

    groups: dict[str, list[Path]] = defaultdict(list)

    for entry in parent.iterdir():
        if entry.is_dir():
            prefix, *_ = entry.name.split("-", 1)
            groups[prefix].append(entry)

    # (Optional) sort the lists for deterministic output
    for folder_list in groups.values():
        folder_list.sort()

    return dict(groups)


def gather_results_method(path_folder: str | Path, method: str, n_j: int, n_m: int, df_results: dict[str, List]) -> dict[str, List]:
    """
    Gather results from all folders in the given path.

    Parameters
    ----------
    path_folder : str | Path
        Path to the directory containing the folders.
    method : str
        Method to gather results for.

    Returns
    -------
    dict[str, pd.DataFrame]
        Dictionary with folder names as keys and DataFrames as values.
    """
    path_folder = os.path.join(path_folder, "sd1")

    path_folder = Path(path_folder).expanduser().resolve()
    if not path_folder.is_dir():
        raise NotADirectoryError(f"{path_folder} is not a directory.")

    folder_dict = group_folders_by_prefix(path_folder)

    norm_score = get_norm_scores()
    for exp_name, folders in folder_dict.items():
        edata_makespan_greedy_results = []
        edata_norm_makespan_greedy_results = []
        edata_gap_greedy_results = []
        edata_norm_gap_greedy_results = []
        edata_runtime_greedy_results = []
        edata_makespan_samp_results = []
        edata_norm_makespan_samp_results = []
        edata_gap_samp_results = []
        edata_norm_gap_samp_results = []
        edata_runtime_samp_results = []
        vdata_makespan_greedy_results = []
        vdata_norm_makespan_greedy_results = []
        vdata_gap_greedy_results = []
        vdata_norm_gap_greedy_results = []
        vdata_runtime_greedy_results = []
        vdata_makespan_samp_results = []
        vdata_norm_makespan_samp_results = []
        vdata_gap_samp_results = []
        vdata_norm_gap_samp_results = []
        vdata_runtime_samp_results = []
        rdata_makespan_greedy_results = []
        rdata_norm_makespan_greedy_results = []

        rdata_gap_greedy_results = []
        rdata_norm_gap_greedy_results = []
        rdata_runtime_greedy_results = []
        rdata_makespan_samp_results = []
        rdata_norm_makespan_samp_results = []
        rdata_gap_samp_results = []
        rdata_norm_gap_samp_results = []

        rdata_runtime_samp_results = []
        mk_makespan_greedy_results = []
        mk_norm_makespan_greedy_results = []

        mk_gap_greedy_results = []
        mk_norm_gap_greedy_results = []
        mk_runtime_greedy_results = []
        mk_makespan_samp_results = []
        mk_norm_makespan_samp_results = []
        mk_gap_samp_results = []
        mk_norm_gap_samp_results = []
        mk_runtime_samp_results = []
        sd1_makespan_greedy_results = []
        sd1_norm_makespan_greedy_results = []
        sd1_gap_greedy_results = []
        sd1_norm_gap_greedy_results = []
        sd1_runtime_greedy_results = []
        sd1_makespan_samp_results = []
        sd1_norm_makespan_samp_results = []
        sd1_gap_samp_results = []
        sd1_norm_gap_samp_results = []
        sd1_runtime_samp_results = []
        assert len(folders) == 4, f"Expected 4 folders for {exp_name}, but found {len(folders)}"
        for folder in folders:
            # print(folder)
            print(exp_name, folder)
            edata_makespan_greedy, edata_makespan_greedy_norm, edata_gap_greedy, edata_gap_greedy_norm, edata_runtime_greedy, edata_makespan_samp, edata_makespan_samp_norm, edata_gap_samp, edata_gap_samp_norm, edata_runtime_samp, \
            vdata_makespan_greedy, vdata_norm_makespan_greedy, vdata_gap_greedy, vdata_norm_gap_greedy, vdata_runtime_greedy, vdata_makespan_samp, vdata_makespan_samp_norm, vdata_gap_samp, vdata_gap_samp_norm, vdata_runtime_samp, \
            rdata_makespan_greedy, rdata_makespan_greedy_norm, rdata_gap_greedy, rdata_gap_greedy_norm, rdata_runtime_greedy, rdata_makespan_samp, rdata_makespan_samp_norm, rdata_gap_samp, rdata_gap_samp_norm, rdata_runtime_samp,\
            mk_makespan_greedy, mk_makespan_greedy_norm, mk_gap_greedy, mk_gap_greedy_norm, mk_runtime_greedy, mk_makespan_samp, mk_makespan_samp_norm, mk_gap_samp, mk_gap_samp_norm,\
            mk_runtime_samp,\
            sd1_makespan_greedy, sd1_makespan_greedy_norm,\
            sd1_gap_greedy,\
            sd1_gap_greedy_norm,\
            sd1_runtime_greedy,\
            sd1_makespan_samp,\
            sd1_makespan_samp_norm,\
            sd1_gap_samp,\
            sd1_gap_samp_norm,\
            sd1_runtime_samp = get_results_comb_bench(folder, exp_name, norm_score, n_j, n_m)

            edata_makespan_greedy_results.append(edata_makespan_greedy)
            edata_norm_makespan_greedy_results.append(edata_makespan_greedy_norm)
            edata_gap_greedy_results.append(edata_gap_greedy)
            edata_norm_gap_greedy_results.append(edata_gap_greedy_norm)
            edata_runtime_greedy_results.append(edata_runtime_greedy)
            edata_makespan_samp_results.append(edata_makespan_samp)
            edata_norm_makespan_samp_results.append(edata_makespan_samp_norm)
            edata_gap_samp_results.append(edata_gap_samp)
            edata_norm_gap_samp_results.append(edata_gap_samp_norm)
            edata_runtime_samp_results.append(edata_runtime_samp)

            vdata_makespan_greedy_results.append(vdata_makespan_greedy)
            vdata_norm_makespan_greedy_results.append(vdata_norm_makespan_greedy)
            vdata_gap_greedy_results.append(vdata_gap_greedy)
            vdata_norm_gap_greedy_results.append(vdata_norm_gap_greedy)
            vdata_runtime_greedy_results.append(vdata_runtime_greedy)
            vdata_makespan_samp_results.append(vdata_makespan_samp)
            vdata_norm_makespan_samp_results.append(vdata_makespan_samp_norm)
            vdata_gap_samp_results.append(vdata_gap_samp)
            vdata_norm_gap_samp_results.append(vdata_gap_samp_norm)
            vdata_runtime_samp_results.append(vdata_runtime_samp)

            rdata_makespan_greedy_results.append(rdata_makespan_greedy)
            rdata_norm_makespan_greedy_results.append(rdata_makespan_greedy_norm)
            rdata_gap_greedy_results.append(rdata_gap_greedy)
            rdata_norm_gap_greedy_results.append(rdata_gap_greedy_norm)
            rdata_runtime_greedy_results.append(rdata_runtime_greedy)
            rdata_makespan_samp_results.append(rdata_makespan_samp)
            rdata_norm_makespan_samp_results.append(rdata_makespan_samp_norm)
            rdata_gap_samp_results.append(rdata_gap_samp)
            rdata_norm_gap_samp_results.append(rdata_gap_samp_norm)
            rdata_runtime_samp_results.append(rdata_runtime_samp)

            mk_makespan_greedy_results.append(mk_makespan_greedy)
            mk_norm_makespan_greedy_results.append(mk_makespan_greedy_norm)
            mk_gap_greedy_results.append(mk_gap_greedy)
            mk_norm_gap_greedy_results.append(mk_gap_greedy_norm)
            mk_runtime_greedy_results.append(mk_runtime_greedy)
            mk_makespan_samp_results.append(mk_makespan_samp)
            mk_norm_makespan_samp_results.append(mk_makespan_samp_norm)
            mk_gap_samp_results.append(mk_gap_samp)
            mk_norm_gap_samp_results.append(mk_gap_samp_norm)
            mk_runtime_samp_results.append(mk_runtime_samp)

            sd1_makespan_greedy_results.append(sd1_makespan_greedy)
            sd1_norm_makespan_greedy_results.append(sd1_makespan_greedy_norm)
            sd1_gap_greedy_results.append(sd1_gap_greedy)
            sd1_norm_gap_greedy_results.append(sd1_gap_greedy_norm)
            sd1_runtime_greedy_results.append(sd1_runtime_greedy)
            sd1_makespan_samp_results.append(sd1_makespan_samp)
            sd1_norm_makespan_samp_results.append(sd1_makespan_samp_norm)
            sd1_gap_samp_results.append(sd1_gap_samp)
            sd1_norm_gap_samp_results.append(sd1_gap_samp_norm)
            sd1_runtime_samp_results.append(sd1_runtime_samp)

        edata_makespan_greedy_mean = np.mean(edata_makespan_greedy_results)
        edata_makespan_greedy_std = np.std(edata_makespan_greedy_results)
        edata_gap_greedy_mean = np.mean(edata_gap_greedy_results)
        edata_gap_greedy_std = np.std(edata_gap_greedy_results)
        edata_runtime_greedy_mean = np.mean(edata_runtime_greedy_results)
        edata_runtime_greedy_std = np.std(edata_runtime_greedy_results)
        edata_makespan_samp_mean = np.mean(edata_makespan_samp_results)
        edata_makespan_samp_std = np.std(edata_makespan_samp_results)
        edata_gap_samp_mean = np.mean(edata_gap_samp_results)
        edata_gap_samp_std = np.std(edata_gap_samp_results)
        edata_runtime_samp_mean = np.mean(edata_runtime_samp_results)
        edata_runtime_samp_std = np.std(edata_runtime_samp_results)
        edata_makespan_greedy_norm_mean = np.mean(edata_norm_makespan_greedy_results)
        edata_makespan_greedy_norm_std = np.std(edata_norm_makespan_greedy_results)
        edata_gap_greedy_norm_mean = np.mean(edata_norm_gap_greedy_results)
        edata_gap_greedy_norm_std = np.std(edata_norm_gap_greedy_results)
        edata_makespan_samp_norm_mean = np.mean(edata_norm_makespan_samp_results)
        edata_makespan_samp_norm_std = np.std(edata_norm_makespan_samp_results)
        edata_gap_samp_norm_mean = np.mean(edata_norm_gap_samp_results)
        edata_gap_samp_norm_std = np.std(edata_norm_gap_samp_results)



        vdata_makespan_greedy_mean = np.mean(vdata_makespan_greedy_results)
        vdata_makespan_greedy_std = np.std(vdata_makespan_greedy_results)
        vdata_gap_greedy_mean = np.mean(vdata_gap_greedy_results)
        vdata_gap_greedy_std = np.std(vdata_gap_greedy_results)
        vdata_runtime_greedy_mean = np.mean(vdata_runtime_greedy_results)
        vdata_runtime_greedy_std = np.std(vdata_runtime_greedy_results)
        vdata_makespan_samp_mean = np.mean(vdata_makespan_samp_results)
        vdata_makespan_samp_std = np.std(vdata_makespan_samp_results)
        vdata_gap_samp_mean = np.mean(vdata_gap_samp_results)
        vdata_gap_samp_std = np.std(vdata_gap_samp_results)
        vdata_runtime_samp_mean = np.mean(vdata_runtime_samp_results)
        vdata_runtime_samp_std = np.std(vdata_runtime_samp_results)
        vdata_makespan_greedy_norm_mean = np.mean(vdata_norm_makespan_greedy_results)
        vdata_makespan_greedy_norm_std = np.std(vdata_norm_makespan_greedy_results)
        vdata_gap_greedy_norm_mean = np.mean(vdata_norm_gap_greedy_results)
        vdata_gap_greedy_norm_std = np.std(vdata_norm_gap_greedy_results)
        vdata_makespan_samp_norm_mean = np.mean(vdata_norm_makespan_samp_results)
        vdata_makespan_samp_norm_std = np.std(vdata_norm_makespan_samp_results)
        vdata_gap_samp_norm_mean = np.mean(vdata_norm_gap_samp_results)
        vdata_gap_samp_norm_std = np.std(vdata_norm_gap_samp_results)


        rdata_makespan_greedy_mean = np.mean(rdata_makespan_greedy_results)
        rdata_makespan_greedy_std = np.std(rdata_makespan_greedy_results)
        rdata_gap_greedy_mean = np.mean(rdata_gap_greedy_results)
        rdata_gap_greedy_std = np.std(rdata_gap_greedy_results)
        rdata_runtime_greedy_mean = np.mean(rdata_runtime_greedy_results)
        rdata_runtime_greedy_std = np.std(rdata_runtime_greedy_results)
        rdata_makespan_samp_mean = np.mean(rdata_makespan_samp_results)
        rdata_makespan_samp_std = np.std(rdata_makespan_samp_results)
        rdata_gap_samp_mean = np.mean(rdata_gap_samp_results)
        rdata_gap_samp_std = np.std(rdata_gap_samp_results)
        rdata_runtime_samp_mean = np.mean(rdata_runtime_samp_results)
        rdata_runtime_samp_std = np.std(rdata_runtime_samp_results)
        rdata_makespan_greedy_norm_mean = np.mean(rdata_norm_makespan_greedy_results)
        rdata_makespan_greedy_norm_std = np.std(rdata_norm_makespan_greedy_results)
        rdata_gap_greedy_norm_mean = np.mean(rdata_norm_gap_greedy_results)
        rdata_gap_greedy_norm_std = np.std(rdata_norm_gap_greedy_results)
        rdata_gap_samp_norm_mean = np.mean(rdata_norm_gap_samp_results)
        rdata_gap_samp_norm_std = np.std(rdata_norm_gap_samp_results)


        mk_makespan_greedy_mean = np.mean(mk_makespan_greedy_results)
        mk_makespan_greedy_std = np.std(mk_makespan_greedy_results)
        mk_gap_greedy_mean = np.mean(mk_gap_greedy_results)
        mk_gap_greedy_std = np.std(mk_gap_greedy_results)
        mk_runtime_greedy_mean = np.mean(mk_runtime_greedy_results)
        mk_runtime_greedy_std = np.std(mk_runtime_greedy_results)
        mk_makespan_samp_mean = np.mean(mk_makespan_samp_results)
        mk_makespan_samp_std = np.std(mk_makespan_samp_results)
        mk_gap_samp_mean = np.mean(mk_gap_samp_results)
        mk_gap_samp_std = np.std(mk_gap_samp_results)
        mk_runtime_samp_mean = np.mean(mk_runtime_samp_results)
        mk_runtime_samp_std = np.std(mk_runtime_samp_results)
        mk_makespan_greedy_norm_mean = np.mean(mk_norm_makespan_greedy_results)
        mk_makespan_greedy_norm_std = np.std(mk_norm_makespan_greedy_results)
        mk_gap_greedy_norm_mean = np.mean(mk_norm_gap_greedy_results)
        mk_gap_greedy_norm_std = np.std(mk_norm_gap_greedy_results)
        mk_makespan_samp_norm_mean = np.mean(mk_norm_makespan_samp_results)
        mk_makespan_samp_norm_std = np.std(mk_norm_makespan_samp_results)
        mk_gap_samp_norm_mean = np.mean(mk_norm_gap_samp_results)
        mk_gap_samp_norm_std = np.std(mk_norm_gap_samp_results)


        sd1_makespan_greedy_mean = np.mean(sd1_makespan_greedy_results)
        sd1_makespan_greedy_std = np.std(sd1_makespan_greedy_results)
        sd1_gap_greedy_mean = np.mean(sd1_gap_greedy_results)
        sd1_gap_greedy_std = np.std(sd1_gap_greedy_results)
        sd1_runtime_greedy_mean = np.mean(sd1_runtime_greedy_results)
        sd1_runtime_greedy_std = np.std(sd1_runtime_greedy_results)
        sd1_makespan_samp_mean = np.mean(sd1_makespan_samp_results)
        sd1_makespan_samp_std = np.std(sd1_makespan_samp_results)

        sd1_gap_samp_mean = np.mean(sd1_gap_samp_results)
        sd1_gap_samp_std = np.std(sd1_gap_samp_results)
        sd1_runtime_samp_mean = np.mean(sd1_runtime_samp_results)
        sd1_runtime_samp_std = np.std(sd1_runtime_samp_results)
        sd1_makespan_greedy_norm_mean = np.mean(sd1_norm_makespan_greedy_results)
        sd1_makespan_greedy_norm_std = np.std(sd1_norm_makespan_greedy_results)
        sd1_gap_greedy_norm_mean = np.mean(sd1_norm_gap_greedy_results)
        sd1_gap_greedy_norm_std = np.std(sd1_norm_gap_greedy_results)
        sd1_makespan_samp_norm_mean = np.mean(sd1_norm_makespan_samp_results)
        sd1_makespan_samp_norm_std = np.std(sd1_norm_makespan_samp_results)
        sd1_gap_samp_norm_mean = np.mean(sd1_norm_gap_samp_results)
        sd1_gap_samp_norm_std = np.std(sd1_norm_gap_samp_results)

        df_results["method"].append(method)
        df_results["dataset"].append(exp_name)
        df_results["n_j"].append(n_j)
        df_results["n_m"].append(n_m)
        df_results.setdefault("makespan_gen_greedy", []).append(sd1_makespan_greedy_mean)
        df_results.setdefault("makespan_gen_greedy_std", []).append(sd1_makespan_greedy_std)
        df_results.setdefault("gap_gen_greedy", []).append(sd1_gap_greedy_mean)
        df_results.setdefault("gap_gen_greedy_std", []).append(sd1_gap_greedy_std)
        df_results.setdefault("runtime_gen_greedy", []).append(sd1_runtime_greedy_mean)
        df_results.setdefault("runtime_gen_greedy_std", []).append(sd1_runtime_greedy_std)
        df_results.setdefault("makespan_gen_greedy_norm", []).append(sd1_makespan_greedy_norm_mean)
        df_results.setdefault("makespan_gen_greedy_norm_std", []).append(sd1_makespan_greedy_norm_std)
        df_results.setdefault("gap_gen_greedy_norm", []).append(sd1_gap_greedy_norm_mean)
        df_results.setdefault("gap_gen_greedy_norm_std", []).append(sd1_gap_greedy_norm_std)

        df_results.setdefault("makespan_gen_samp", []).append(sd1_makespan_samp_mean)
        df_results.setdefault("makespan_gen_samp_std", []).append(sd1_makespan_samp_std)
        df_results.setdefault("gap_gen_samp", []).append(sd1_gap_samp_mean)
        df_results.setdefault("gap_gen_samp_std", []).append(sd1_gap_samp_std)
        df_results.setdefault("runtime_gen_samp", []).append(sd1_runtime_samp_mean)
        df_results.setdefault("runtime_gen_samp_std", []).append(sd1_runtime_samp_std)
        df_results.setdefault("makespan_gen_samp_norm", []).append(sd1_makespan_samp_norm_mean)
        df_results.setdefault("makespan_gen_samp_norm_std", []).append(sd1_makespan_samp_norm_std)
        df_results.setdefault("gap_gen_samp_norm", []).append(sd1_gap_samp_norm_mean)
        df_results.setdefault("gap_gen_samp_norm_std", []).append(sd1_gap_samp_norm_std)

        df_results.setdefault("makespan_vdata_greedy", []).append(vdata_makespan_greedy_mean)
        df_results.setdefault("makespan_vdata_greedy_std", []).append(vdata_makespan_greedy_std)
        df_results.setdefault("gap_vdata_greedy", []).append(vdata_gap_greedy_mean)
        df_results.setdefault("gap_vdata_greedy_std", []).append(vdata_gap_greedy_std)
        df_results.setdefault("runtime_vdata_greedy", []).append(vdata_runtime_greedy_mean)
        df_results.setdefault("runtime_vdata_greedy_std", []).append(vdata_runtime_greedy_std)
        df_results.setdefault("makespan_vdata_greedy_norm", []).append(vdata_makespan_greedy_norm_mean)
        df_results.setdefault("makespan_vdata_greedy_norm_std", []).append(vdata_makespan_greedy_norm_std)
        df_results.setdefault("gap_vdata_greedy_norm", []).append(vdata_gap_greedy_norm_mean)
        df_results.setdefault("gap_vdata_greedy_norm_std", []).append(vdata_gap_greedy_norm_std)

        df_results.setdefault("makespan_vdata_samp", []).append(vdata_makespan_samp_mean)
        df_results.setdefault("makespan_vdata_samp_std", []).append(vdata_makespan_samp_std)
        df_results.setdefault("gap_vdata_samp", []).append(vdata_gap_samp_mean)
        df_results.setdefault("gap_vdata_samp_std", []).append(vdata_gap_samp_std)
        df_results.setdefault("runtime_vdata_samp", []).append(vdata_runtime_samp_mean)
        df_results.setdefault("runtime_vdata_samp_std", []).append(vdata_runtime_samp_std)
        df_results.setdefault("makespan_vdata_samp_norm", []).append(vdata_makespan_samp_norm_mean)
        df_results.setdefault("makespan_vdata_samp_norm_std", []).append(vdata_makespan_samp_norm_std)
        df_results.setdefault("gap_vdata_samp_norm", []).append(vdata_gap_samp_norm_mean)
        df_results.setdefault("gap_vdata_samp_norm_std", []).append(vdata_gap_samp_norm_std)

        df_results.setdefault("makespan_rdata_greedy", []).append(rdata_makespan_greedy_mean)
        df_results.setdefault("makespan_rdata_greedy_std", []).append(rdata_makespan_greedy_std)
        df_results.setdefault("gap_rdata_greedy", []).append(rdata_gap_greedy_mean)
        df_results.setdefault("gap_rdata_greedy_std", []).append(rdata_gap_greedy_std)
        df_results.setdefault("runtime_rdata_greedy", []).append(rdata_runtime_greedy_mean)
        df_results.setdefault("runtime_rdata_greedy_std", []).append(rdata_runtime_greedy_std)
        df_results.setdefault("makespan_rdata_greedy_norm", []).append(rdata_makespan_greedy_norm_mean)
        df_results.setdefault("makespan_rdata_greedy_norm_std", []).append(rdata_makespan_greedy_norm_std)
        df_results.setdefault("gap_rdata_greedy_norm", []).append(rdata_gap_greedy_norm_mean)
        df_results.setdefault("gap_rdata_greedy_norm_std", []).append(rdata_gap_greedy_norm_std)

        df_results.setdefault("makespan_rdata_samp", []).append(rdata_makespan_samp_mean)
        df_results.setdefault("makespan_rdata_samp_std", []).append(rdata_makespan_samp_std)
        df_results.setdefault("gap_rdata_samp", []).append(rdata_gap_samp_mean)
        df_results.setdefault("gap_rdata_samp_std", []).append(rdata_gap_samp_std)
        df_results.setdefault("runtime_rdata_samp", []).append(rdata_runtime_samp_mean)
        df_results.setdefault("runtime_rdata_samp_std", []).append(rdata_runtime_samp_std)
        df_results.setdefault("makespan_rdata_samp_norm", []).append(rdata_makespan_greedy_norm_mean)
        df_results.setdefault("makespan_rdata_samp_norm_std", []).append(rdata_makespan_greedy_norm_std)
        df_results.setdefault("gap_rdata_samp_norm", []).append(rdata_gap_samp_norm_mean)
        df_results.setdefault("gap_rdata_samp_norm_std", []).append(rdata_gap_samp_norm_std)

        df_results.setdefault("makespan_edata_greedy", []).append(edata_makespan_greedy_mean)
        df_results.setdefault("makespan_edata_greedy_std", []).append(edata_makespan_greedy_std)
        df_results.setdefault("gap_edata_greedy", []).append(edata_gap_greedy_mean)
        df_results.setdefault("gap_edata_greedy_std", []).append(edata_gap_greedy_std)
        df_results.setdefault("runtime_edata_greedy", []).append(edata_runtime_greedy_mean)
        df_results.setdefault("runtime_edata_greedy_std", []).append(edata_runtime_greedy_std)
        df_results.setdefault("makespan_edata_greedy_norm", []).append(edata_makespan_greedy_norm_mean)
        df_results.setdefault("makespan_edata_greedy_norm_std", []).append(edata_makespan_greedy_norm_std)
        df_results.setdefault("gap_edata_greedy_norm", []).append(edata_gap_greedy_norm_mean)
        df_results.setdefault("gap_edata_greedy_norm_std", []).append(edata_gap_greedy_norm_std)

        df_results.setdefault("makespan_edata_samp", []).append(edata_makespan_samp_mean)
        df_results.setdefault("makespan_edata_samp_std", []).append(edata_makespan_samp_std)
        df_results.setdefault("gap_edata_samp", []).append(edata_gap_samp_mean)
        df_results.setdefault("gap_edata_samp_std", []).append(edata_gap_samp_std)
        df_results.setdefault("runtime_edata_samp", []).append(edata_runtime_samp_mean)
        df_results.setdefault("runtime_edata_samp_std", []).append(edata_runtime_samp_std)
        df_results.setdefault("makespan_edata_samp_norm", []).append(edata_makespan_samp_norm_mean)
        df_results.setdefault("makespan_edata_samp_norm_std", []).append(edata_makespan_samp_norm_std)
        df_results.setdefault("gap_edata_samp_norm", []).append(edata_gap_samp_norm_mean)
        df_results.setdefault("gap_edata_samp_norm_std", []).append(edata_gap_samp_norm_std)

        df_results.setdefault("makespan_mk_greedy", []).append(mk_makespan_greedy_mean)
        df_results.setdefault("makespan_mk_greedy_std", []).append(mk_makespan_greedy_std)
        df_results.setdefault("gap_mk_greedy", []).append(mk_gap_greedy_mean)
        df_results.setdefault("gap_mk_greedy_std", []).append(mk_gap_greedy_std)
        df_results.setdefault("runtime_mk_greedy", []).append(mk_runtime_greedy_mean)
        df_results.setdefault("runtime_mk_greedy_std", []).append(mk_runtime_greedy_std)
        df_results.setdefault("makespan_mk_greedy_norm", []).append(mk_makespan_greedy_norm_mean)
        df_results.setdefault("makespan_mk_greedy_norm_std", []).append(mk_makespan_greedy_norm_std)
        df_results.setdefault("gap_mk_greedy_norm", []).append(mk_gap_greedy_norm_mean)
        df_results.setdefault("gap_mk_greedy_norm_std", []).append(mk_gap_greedy_norm_std)

        df_results.setdefault("makespan_mk_samp", []).append(mk_makespan_samp_mean)
        df_results.setdefault("makespan_mk_samp_std", []).append(mk_makespan_samp_std)
        df_results.setdefault("gap_mk_samp", []).append(mk_gap_samp_mean)
        df_results.setdefault("gap_mk_samp_std", []).append(mk_gap_samp_std)
        df_results.setdefault("runtime_mk_samp", []).append(mk_runtime_samp_mean)
        df_results.setdefault("runtime_mk_samp_std", []).append(mk_runtime_samp_std)
        df_results.setdefault("makespan_mk_samp_norm", []).append(mk_makespan_samp_norm_mean)
        df_results.setdefault("makespan_mk_samp_norm_std", []).append(mk_makespan_samp_norm_std)
        df_results.setdefault("gap_mk_samp_norm", []).append(mk_gap_samp_norm_mean)
        df_results.setdefault("gap_mk_samp_norm_std", []).append(mk_gap_samp_norm_std)


    return df_results

def main_start_func():
    df_results = {
        "method": [],
        "dataset": [],
        "n_j": [],
        "n_m": [],
        "makespan_gen_greedy": [],
        "makespan_gen_greedy_std": [],
        "makespan_gen_greedy_norm": [],
        "makespan_gen_greedy_norm_std": [],
        "gap_gen_greedy": [],
        "gap_gen_greedy_std": [],
        "gap_gen_greedy_norm": [],
        "gap_gen_greedy_norm_std": [],
        "runtime_gen_greedy": [],
        "runtime_gen_greedy_std": [],
        "makespan_gen_samp": [],
        "makespan_gen_samp_std": [],
        "makespan_gen_samp_norm": [],
        "makespan_gen_samp_norm_std": [],
        "gap_gen_samp": [],
        "gap_gen_samp_std": [],
        "gap_gen_samp_norm": [],
        "gap_gen_samp_norm_std": [],
        "runtime_gen_samp": [],
        "runtime_gen_samp_std": [],
        "makespan_vdata_greedy": [],
        "makespan_vdata_greedy_std": [],
        "makespan_vdata_greedy_norm": [],
        "makespan_vdata_greedy_norm_std": [],
        "gap_vdata_greedy": [],
        "gap_vdata_greedy_std": [],
        "gap_vdata_greedy_norm": [],
        "gap_vdata_greedy_norm_std": [],
        "runtime_vdata_greedy": [],
        "runtime_vdata_greedy_std": [],
        "makespan_vdata_samp": [],
        "makespan_vdata_samp_std": [],
        "makespan_vdata_samp_norm": [],
        "makespan_vdata_samp_norm_std": [],
        "gap_vdata_samp": [],
        "gap_vdata_samp_std": [],
        "gap_vdata_samp_norm": [],
        "gap_vdata_samp_norm_std": [],
        "runtime_vdata_samp": [],
        "runtime_vdata_samp_std": [],
        "makespan_rdata_greedy": [],
        "makespan_rdata_greedy_std": [],
        "makespan_rdata_greedy_norm": [],
        "makespan_rdata_greedy_norm_std": [],
        "gap_rdata_greedy": [],
        "gap_rdata_greedy_std": [],
        "gap_rdata_greedy_norm": [],
        "gap_rdata_greedy_norm_std": [],
        "runtime_rdata_greedy": [],
        "runtime_rdata_greedy_std": [],
        "makespan_rdata_samp": [],
        "makespan_rdata_samp_std": [],
        "makespan_rdata_samp_norm": [],
        "makespan_rdata_samp_norm_std": [],
        "gap_rdata_samp": [],
        "gap_rdata_samp_std": [],
        "gap_rdata_samp_norm": [],
        "gap_rdata_samp_norm_std": [],
        "runtime_rdata_samp": [],
        "runtime_rdata_samp_std": [],
        "makespan_edata_greedy": [],
        "makespan_edata_greedy_std": [],
        "makespan_edata_greedy_norm": [],
        "makespan_edata_greedy_norm_std": [],
        "gap_edata_greedy": [],
        "gap_edata_greedy_std": [],
        "gap_edata_greedy_norm": [],
        "gap_edata_greedy_norm_std": [],
        "runtime_edata_greedy": [],
        "runtime_edata_greedy_std": [],
        "makespan_edata_samp": [],
        "makespan_edata_samp_std": [],
        "makespan_edata_samp_norm": [],
        "makespan_edata_samp_norm_std": [],
        "gap_edata_samp": [],
        "gap_edata_samp_std": [],
        "gap_edata_samp_norm": [],
        "gap_edata_samp_norm_std": [],
        "runtime_edata_samp": [],
        "runtime_edata_samp_std": [],
        "makespan_mk_greedy": [],
        "makespan_mk_greedy_std": [],
        "makespan_mk_greedy_norm": [],
        "makespan_mk_greedy_norm_std": [],
        "gap_mk_greedy": [],
        "gap_mk_greedy_std": [],
        "gap_mk_greedy_norm": [],
        "gap_mk_greedy_norm_std": [],
        "runtime_mk_greedy": [],
        "runtime_mk_greedy_std": [],

        "makespan_mk_samp": [],
        "makespan_mk_samp_std": [],
        "makespan_mk_samp_norm": [],
        "makespan_mk_samp_norm_std": [],
        "gap_mk_samp": [],
        "gap_mk_samp_std": [],
        "gap_mk_samp_norm": [],
        "gap_mk_samp_norm_std": [],
        "runtime_mk_samp": [],
        "runtime_mk_samp_std": [],
    }
    methods = ["dmSAC", "mQRDQN", "IQL", "CDQAC"]
    dataset_size = [(10, 5), (15, 10), (20, 10)]
    for n_j, n_m in dataset_size:
        for method in methods:
            path = f"./checkpoints_exp/dataset_exp_{n_j}_{n_m}/{method}"
            df_results = gather_results_method(path, method, n_j, n_m, df_results)
    print(df_results)
    df_results = pd.DataFrame(df_results)
    df_results.to_csv("./results_datasets/all_results.csv", index=False)
    print(df_results)

main_start_func()

# def collect_results(path: str | Path, rename_dict_keys: Optional[dict] = None, save_path: Optional[str] = None) -> dict[str, dict[str, pd.DataFrame]]:
#     """
#     Collect results from all folders in the given path.
#
#     Parameters
#     ----------
#     path : str | Path
#         Path to the directory containing the folders.
#
#     Returns
#     -------
#     dict[str, dict[str, pd.DataFrame]]
#         Dictionary with folder names as keys and dictionaries of DataFrames as values.
#     """
#     path = os.path.join(path, "CDQAC", "sd1")
#     path = Path(path).expanduser().resolve()
#
#     if not path.is_dir():
#         raise NotADirectoryError(f"{path} is not a directory.")
#
#     df_results = {
#         "exp_name": [],
#         "gap_gen_greedy": [],
#         "gap_gen_greedy_std": [],
#         "gap_gen_samp": [],
#         "gap_gen_samp_std": [],
#         "gap_bench_greedy": [],
#         "gap_bench_greedy_std": [],
#         "gap_bench_samp": [],
#         "gap_bench_samp_std": [],
#     }
#     folder_dict = group_folders_by_prefix(path)
#     if rename_dict_keys is not None:
#         folder_dict = {rename_dict_keys.get(k, k): v for k, v in folder_dict.items()}
#     for exp_name, folders in folder_dict.items():
#         # print(exp_name)
#         gap_gen_greedy = []
#         gap_gen_samp = []
#         gap_bench_greedy = []
#         gap_bench_samp = []
#         assert len(folders) == 4, f"Expected 4 folders for {exp_name}, but found {len(folders)}"
#         for folder in folders:
#             # print(folder)
#             gap_gen_g, gap_gen_s, gap_bench_g, gap_bench_s = get_results_comb_bench(folder)
#             gap_gen_greedy.append(gap_gen_g)
#             gap_gen_samp.append(gap_gen_s)
#             gap_bench_greedy.append(gap_bench_g)
#             gap_bench_samp.append(gap_bench_s)
#         gap_gen_greedy_mean = np.mean(gap_gen_greedy)
#         gap_gen_greedy_std = np.std(gap_gen_greedy)
#         gap_gen_samp_mean = np.mean(gap_gen_samp)
#         gap_gen_samp_std = np.std(gap_gen_samp)
#         gap_bench_greedy_mean = np.mean(gap_bench_greedy)
#         gap_bench_greedy_std = np.std(gap_bench_greedy)
#         gap_bench_samp_mean = np.mean(gap_bench_samp)
#         gap_bench_samp_std = np.std(gap_bench_samp)
#         df_results["exp_name"].append(int(exp_name))
#         df_results["gap_gen_greedy"].append(gap_gen_greedy_mean)
#         df_results["gap_gen_greedy_std"].append(gap_gen_greedy_std)
#         df_results["gap_gen_samp"].append(gap_gen_samp_mean)
#         df_results["gap_gen_samp_std"].append(gap_gen_samp_std)
#         df_results["gap_bench_greedy"].append(gap_bench_greedy_mean)
#         df_results["gap_bench_greedy_std"].append(gap_bench_greedy_std)
#         df_results["gap_bench_samp"].append(gap_bench_samp_mean)
#         df_results["gap_bench_samp_std"].append(gap_bench_samp_std)
#     df_results = pd.DataFrame(df_results)
#     # df_results[]
#     df_results = df_results.sort_values(by=['exp_name'], ascending=True)
#     # print(df_results)
#     if save_path is not None:
#         df_results.to_csv(save_path, index=False)
#         # return




NameError: name 'Path' is not defined

In [51]:
import pandas as pd
from pathlib import Path
import textwrap

# ----------------------------------------------------------------------
# Helper to format "mean ± std" with a chosen number of decimals
# ----------------------------------------------------------------------
def _pm(mean: float, std: float, digits: int = 2) -> str:
    """Format 'mean ± std' with the requested precision."""
    return f"{mean:.{digits}f}$\\pm${std:.{digits}f}"

def _pm(mean: float, std: float, digits: int = 2) -> str:
    return f"{mean:.{digits}f}$\\pm${std:.{digits}f}"

# ----------------------------------------------------------------------
# Main generator  ------------------------------------------------------
# ----------------------------------------------------------------------
def generate_table_grouped(
    df: pd.DataFrame,
    n_j: int,
    n_m: int,
    digits: int = 2,
    path_out: Path | None = None
) -> str:
    """
    Return (and optionally write) a LaTeX table that shows greedy (g) and
    sampling (s) results side-by-side, with rows grouped by *dataset*.
    Columns n_j and n_m are omitted from the final table.
    """

    # ---------- benchmark sets (CSV prefix  → short label) -------------
    sets = [
        ("vdata", "V"),
        ("rdata", "R"),
        ("edata", "E"),
        ("mk",    "MK"),
        ("gen",   "G"),   # change to ('generated', 'G') if needed
    ]

    # ---------- evaluation modes (CSV suffix → header tag) -------------
    modes = [
        ("greedy", "g"),
        ("samp",   "s"),
    ]

    # ---------- subset -------------------------------------------------
    sub = df.query("n_j == @n_j and n_m == @n_m").copy()
    if sub.empty:
        raise ValueError(f"No rows for n_j={n_j}, n_m={n_m}")

    # ---------- prepare header ----------------------------------------
    n_extra = 4 * len(sets)               # 4 metric columns per set
    col_spec = "ll" + "c" * n_extra       # 'Train set' + 'Method' + metrics

    head = textwrap.dedent(f"""
        \\begin{{table*}}[h]
        \\centering
        \\caption{{Mean $\\pm$ std of Gap and normalised Gap on five benchmark
        sets (greedy (g) and sampling (s)) for $n_j={n_j}$, $n_m={n_m}$.}}
        \\resizebox{{\\textwidth}}{{!}}{{%
        \\begin{{tabular}}{{{col_spec}}}
        \\toprule
        \\multirow{{2}}{{*}}{{Train set}} &
        \\multirow{{2}}{{*}}{{Method}}""").rstrip()

    for _, label in sets:
        head += f" & \\multicolumn{{4}}{{c}}{{{label}}}"
    head += " \\\\\n & "          # continue header row 2
    head += "& ".join([""] * 1)   # just to align ampersands
    head += " & "                 # align after 'Method'
    for _ in sets:
        head += "Gap$_{g}$ & Norm$_{g}$ & Gap$_{s}$ & Norm$_{s}$ & "
    head = head.rstrip("& ") + r"\\\midrule" + "\n"

    # ---------- body with grouping ------------------------------------
    body_lines = []
    sub = sub.sort_values(["dataset", "method"])
    for dataset, grp in sub.groupby("dataset", sort=False):
        row_span = len(grp)
        first = True
        for _, row in grp.iterrows():
            line = ""
            if first:
                line += f"\\multirow{{{row_span}}}{{*}}{{{dataset}}} "
                first = False
            else:
                line += " "  # placeholder cell

            line += f"& {row['method']} "
            for prefix, _ in sets:
                for mode, _tag in modes:
                    gap_col       = f"gap_{prefix}_{mode}"
                    gap_std_col   = f"{gap_col}_std"
                    norm_col      = f"{gap_col}_norm"
                    norm_std_col  = f"{norm_col}_std"

                    gap  = _pm(row[gap_col],  row[gap_std_col],  digits)
                    norm = _pm(row[norm_col], row[norm_std_col], digits)
                    line += f"& {gap} & {norm} "
            body_lines.append(line + r"\\")

    # ---------- footer ------------------------------------------------
    tail = textwrap.dedent(r"""
        \bottomrule
        \end{tabular}}
        \end{table*}""")

    tex = "\n".join([head, *body_lines, tail])
    if path_out is not None:
        path_out.write_text(tex, encoding="utf-8")
    return tex
df_results = pd.read_csv("./results_datasets/all_results.csv")
test = generate_table_grouped(
    df_results,
    n_j=10,
    n_m=5,
    digits=2,
    path_out=Path("./results_datasets/latex_table.tex"),
)




In [55]:
csv_file   = "all_results.csv"              # path to your results
eval_keys  = ["gen", "mk", "edata", "vdata", "rdata"]
eval_names = ["Generated", "MK", "Edata", "Vdata", "Rdata"]
train_sets = ["dispatching", "dis_pop", "population", "random"]  # row-block order
methods    = ["dmSAC", "mQRDQN", "IQL", "CDQAC"]                 # CDQAC last
policies   = ["greedy", "samp"]                                  # outer groups

def fmt(mean, std):
    """Format as 12.34 ± 1.23 for LaTeX."""
    return f"{mean:.1f}\\%$\\pm${std:.1f}\\%"

# ---------- read & aggregate ------------------------------------------------
df  =  df_results
agg = (
    df.groupby(["dataset", "method"])
      .mean(numeric_only=True)   # mean over instance sizes / seeds
      .reset_index()
)

# ---------- build LaTeX -----------------------------------------------------
lines = [
    r"\begin{table*}[ht]",
    r"\centering",
    r"\caption{Gap (\%) with standard deviation for every training dataset, "
    r"grouped by policy (Greedy / Sampling).}",
    r"\resizebox{\textwidth}{!}{%",
    r"\begin{tabular}{l" + "c" * 8 + "}",
    r"\toprule",
    # outer policy groups ----------------------------------------------------
    " & " + r"\multicolumn{4}{c}{\textbf{Greedy}}" +
    " & " + r"\multicolumn{4}{c}{\textbf{Sampling}}" + r"\\",
    # inner method names -----------------------------------------------------
    "Train / Eval & " +
    " & ".join(methods + methods) + r"\\",
    r"\midrule",
]

for tr in train_sets:
    first_row = True
    for ek, en in zip(eval_keys, eval_names):
        cells = []
        if first_row:                         # multirow label for training set
            cells.append(fr"\multirow{{5}}{{*}}{{\texttt{{{tr}}}}}")
            first_row = False
        else:
            cells.append("")                  # empty cell below multirow
        cells.append(en)                      # evaluation set label

        # metrics: Greedy block then Sampling block --------------------------
        for pol in policies:
            for m in methods:
                mean = agg.loc[
                    (agg["dataset"] == tr) & (agg["method"] == m),
                    f"gap_{ek}_{pol}"
                ].iloc[0]
                std  = agg.loc[
                    (agg["dataset"] == tr) & (agg["method"] == m),
                    f"gap_{ek}_{pol}_std"
                ].iloc[0]
                cells.append(fmt(mean, std))

        lines.append(" & ".join(cells) + r"\\")

    lines.append(r"\midrule")

# replace the last \midrule with \bottomrule ---------------------------------
lines[-1] = r"\bottomrule"
lines.extend([r"\end{tabular}%", r"}", r"\end{table*}"])

# ----------- output ---------------------------------------------------------
latex_table = "\n".join(lines)
print(latex_table)

\begin{table*}[ht]
\centering
\caption{Gap (\%) with standard deviation for every training dataset, grouped by policy (Greedy / Sampling).}
\resizebox{\textwidth}{!}{%
\begin{tabular}{lcccccccc}
\toprule
 & \multicolumn{4}{c}{\textbf{Greedy}} & \multicolumn{4}{c}{\textbf{Sampling}}\\
Train / Eval & dmSAC & mQRDQN & IQL & CDQAC & dmSAC & mQRDQN & IQL & CDQAC\\
\midrule
\multirow{5}{*}{\texttt{dispatching}} & Generated & 15.7\%$\pm$1.7\% & 20.1\%$\pm$2.8\% & 14.1\%$\pm$0.7\% & 11.0\%$\pm$2.3\% & 9.0\%$\pm$0.4\% & 16.5\%$\pm$0.1\% & 8.5\%$\pm$0.2\% & 5.5\%$\pm$1.2\%\\
 & MK & 46.4\%$\pm$5.3\% & 26.5\%$\pm$2.7\% & 39.3\%$\pm$3.6\% & 14.6\%$\pm$0.9\% & 26.7\%$\pm$2.5\% & 25.5\%$\pm$0.6\% & 22.2\%$\pm$1.3\% & 9.2\%$\pm$0.3\%\\
 & Edata & 26.3\%$\pm$4.3\% & 29.0\%$\pm$2.0\% & 22.9\%$\pm$2.2\% & 18.1\%$\pm$2.6\% & 12.6\%$\pm$2.1\% & 12.5\%$\pm$0.2\% & 11.0\%$\pm$0.8\% & 10.6\%$\pm$1.0\%\\
 & Vdata & 7.8\%$\pm$1.5\% & 13.5\%$\pm$2.1\% & 6.0\%$\pm$0.5\% & 5.4\%$\pm$1.6\% & 1.6\%$\pm$0.4\% & 3.4\

In [8]:
import pandas as pd
df_results = pd.read_csv("./results_datasets/all_results.csv")

eval_keys   = ["gen", "mk", "edata", "vdata", "rdata"]
eval_names  = ["Generated", "MK", "Edata", "Vdata", "Rdata"]
train_sets  = ["dispatching", "dis_pop", "population", "random"]      # row–block order
methods     = ["dmSAC", "mQRDQN", "IQL", "CDQAC"]                     # CDQAC last
policies    = ["greedy", "samp"]                                      # outer groups
train_sizes = [(10, 5), (15, 10), (20, 10)]                              # NEW

def fmt(mean, std):
    """Format 12.34 ± 1.23 as required for LaTeX."""
    return rf"{mean:.1f}\%\,$\pm$\,{std:.1f}\%"

# ---------------------------------------------------------------------------
# 1. Aggregate without mixing training sizes
# ---------------------------------------------------------------------------
# Your dataframe must contain a column named *train_size* holding strings
# "10×5", "15×10", "20×10".  Adapt the name if necessary.
agg = (
    df_results
      .groupby(["n_j", "n_m", "dataset", "method"])
      .mean(numeric_only=True)        # mean over seeds only
      .reset_index()
)

# ---------------------------------------------------------------------------
# 2. Build one table per training size
# ---------------------------------------------------------------------------
tables = []                                     # collect the complete tables
for n_j, n_m in train_sizes:
    lines = [
        r"\begin{table*}[ht]",
        r"\centering",
        rf"\caption{{Gap (\%) ± s.d. when the policy is \textbf{{trained on size {ts}}}. "
        r"Rows show the evaluation benchmark; columns are methods, "
        r"grouped by policy (Greedy / Sampling).}}",
        r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{l" + "c"*8 + "}",
        r"\toprule",
        " & " + r"\multicolumn{4}{c}{\textbf{Greedy}}"
        " & " + r"\multicolumn{4}{c}{\textbf{Sampling}}" + r"\\",
        # inner-block method names
        r"Train / Eval & " + " & ".join(methods + methods) + r"\\",
        r"\midrule",
    ]

    # --- body --------------------------------------------------------------
    for tr in train_sets:
        first_row = True
        for ek, en in zip(eval_keys, eval_names):
            cells = []
            # multi-row label for the training set
            if first_row:
                cells.append(rf"\multirow{{5}}{{*}}{{\texttt{{{tr}}}}}")
                first_row = False
            else:
                cells.append("")                     # cell under the multi-row
            cells.append(en)                         # evaluation-set label

            # Greedy block then Sampling block
            for pol in policies:
                for m in methods:
                    r = agg.loc[
                        (agg["n_j"] == n_j)
                        & (agg["n_m"] == n_m)
                        & (agg["dataset"]    == tr)
                        & (agg["method"]     == m),
                        [f"gap_{ek}_{pol}", f"gap_{ek}_{pol}_std"]
                    ].iloc[0]
                    cells.append(fmt(r[0], r[1]))

            lines.append(" & ".join(cells) + r"\\")

        lines.append(r"\midrule")

    # swap the last \midrule for \bottomrule and finish up
    lines[-1] = r"\bottomrule"
    lines.extend([r"\end{tabular}%", r"}", r"\end{table*}"])
    tables.append("\n".join(lines))

# ---------------------------------------------------------------------------
# 3. Write each LaTeX table to file (optional)
# ---------------------------------------------------------------------------
print(tables)
for ts, tex in zip(train_sizes, tables):
    n_j, n_m = ts
    with open(f"results_{n_j}_{n_m}.tex", "w") as f:
        f.write(tex)

C:\Users\jesse\AppData\Local\Temp\ipykernel_46368\2760998164.py:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  cells.append(fmt(r[0], r[1]))
C:\Users\jesse\AppData\Local\Temp\ipykernel_46368\2760998164.py:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  cells.append(fmt(r[0], r[1]))
C:\Users\jesse\AppData\Local\Temp\ipykernel_46368\2760998164.py:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  cells.append(fmt(r[0], r[1]))
C:\U

['\\begin{table*}[ht]\n\\centering\n\\caption{Gap (\\%) ± s.d. when the policy is \\textbf{trained on size (10, 5)}. Rows show the evaluation benchmark; columns are methods, grouped by policy (Greedy / Sampling).}}\n\\resizebox{\\textwidth}{!}{%\n\\begin{tabular}{lcccccccc}\n\\toprule\n & \\multicolumn{4}{c}{\\textbf{Greedy}} & \\multicolumn{4}{c}{\\textbf{Sampling}}\\\\\nTrain / Eval & dmSAC & mQRDQN & IQL & CDQAC & dmSAC & mQRDQN & IQL & CDQAC\\\\\n\\midrule\n\\multirow{5}{*}{\\texttt{dispatching}} & Generated & 15.3\\%\\,$\\pm$\\,0.9\\% & 15.4\\%\\,$\\pm$\\,1.2\\% & 15.6\\%\\,$\\pm$\\,0.5\\% & 11.5\\%\\,$\\pm$\\,0.4\\% & 8.2\\%\\,$\\pm$\\,0.1\\% & 14.4\\%\\,$\\pm$\\,0.1\\% & 8.1\\%\\,$\\pm$\\,0.2\\% & 5.6\\%\\,$\\pm$\\,0.1\\%\\\\\n & MK & 43.7\\%\\,$\\pm$\\,5.4\\% & 22.8\\%\\,$\\pm$\\,3.8\\% & 41.8\\%\\,$\\pm$\\,3.9\\% & 12.4\\%\\,$\\pm$\\,1.4\\% & 23.2\\%\\,$\\pm$\\,3.4\\% & 25.1\\%\\,$\\pm$\\,0.3\\% & 21.9\\%\\,$\\pm$\\,1.1\\% & 8.3\\%\\,$\\pm$\\,0.1\\%\\\\\n & Edata & 22.2\\%\\,$